In [2]:
print("Kernel works")

Kernel works


In [3]:
import pandas as pd
from rdkit import Chem

df = pd.read_csv ('/home/susan/mof-co2-adsorption/data/processed/df_chem.csv')
print(df.columns)
print("Dataset shape:", df.shape)
print("Missing MOFID:", df["mofid"].isna().sum())
print("Unique MOFID:", df["mofid"].nunique())


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')
Dataset shape: (27706, 11)
Missing MOFID: 0
Unique MOFID: 24958


In [4]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


---
we want to answer three questions:

- How many extra occurrences are caused by repeated MOFIDs?
- How many different MOFID strings appear more than once?
- How many times can one MOFID occur?

Step 1 — Count occurrence of each MOFID.


In [5]:
mofid_counts = df["mofid"].value_counts()
# Keep only MOFIDs that appear more than once
repeated_mofids = mofid_counts[mofid_counts > 1]

print("Total rows:", len(df))
print("Unique MOFIDs:", df["mofid"].nunique())
print("Repeated occurrences:", df["mofid"].duplicated().sum())

print(mofid_counts.head())


Total rows: 27706
Unique MOFIDs: 24958
Repeated occurrences: 2748
mofid
* MOFid-v1.NA.NA                                                                         1259
* MOFid-v1.NA.NAno_mof                                                                    495
N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERROR.cat0                                     39
N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERROR.cat0                                     32
[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21
Name: count, dtype: int64


| Output                                | Meaning                                                                                                                                             |
| ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Total rows: 27,706**                |  dataset contains 27,706 MOF records with a non-null value in the `mofid` column.                                                               |
| **Unique MOFIDs: 24,958**             | There are 24,958 different `mofid` strings among those 27,706 rows.                                                                                 |
| **Extra repeated occurrences: 2,748** | After keeping the first occurrence of every MOFID, there are 2,748 additional occurrences of already-seen MOFID strings. This is `27,706 − 24,958`. |
| **Unique MOFIDs that repeat: 616**    | There are 616 different MOFID strings that occur at least twice.                                                                                    |
| **Maximum occurrence: 1,259**         | The most frequently occurring MOFID string appears in 1,259 rows.                                                                                   |


Total rows: 27706
Unique MOFIDs: 24958
Extra repeated occurrences: 2748
Unique MOFIDs that repeat: 616 `number of repeated MOFID types`
Maximum occurrence of one MOFID: 1259 `highest frequency of one MOFID type`
mofid

---


`* MOFid-v1.NA.NA    1259`
means this exact string occurs in 1,259 rows. NA indicates that a normal MOFID representation was not available/generated, so these are not 1,259 copies of the same chemical structure.

---
*MOFid-v1.NA.NAno_mof   495*
means this exact status-like string occurs 495 times. Again, this is not a normal chemical MOFID.

---

*N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERRORcat0  39*

means the exact same MOFID string occurs 39 times. Unlike the NA entries, it contains chemical fragments (N#N, linker, Zn) but its MOFID metadata contains ERROR.

---

*N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERRORcat0   32*
is another specific chemical representation occurring 32 times, also carrying an ERROR status.

---
*[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21*
occurs 21 times. This looks different because pcu is a topology designation rather than an NA or ERROR marker.

*The key distinction is:*
27,706 rows ≠ 27,706 unique MOFIDs. There are 24,958 unique MOFID strings, and the repetition is strongly influenced by placeholder/error values such as the 1,259 NA records.

removing both
* MOFid-v1.NA.NA
          ^^^^^^^^^^

* MOFid-v1.NA.NAno_mof
          ^^^^^^^^^^

In [6]:
# removing both
# MOFid-v1.NA.NA
#  MOFid-v1.NA.NAno_mof
df = df[
    ~df["mofid"].str.contains("MOFid-v1.NA", na=False)
].copy()

print("Remaining rows:", len(df))
#Then verify:
print(df.shape)
print(df["mofid"].str.contains("MOFid-v1.NA", na=False).sum())


Remaining rows: 25952
(25952, 11)
0


In [7]:
# quantify only the ERROR records:
error_count = df["mofid"].str.contains(
    "MOFid-v1.ERROR", na=False
).sum()

print("ERROR-type MOFIDs:", error_count)

# take Take one ERROR MOFID to check it works with rdkit
error_example = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"].iloc[0]

print(error_example)

# extract it :
chemical_smiles = error_example.split(" ")[0]
print(chemical_smiles)

# Test with RDKIT 
mol = Chem.MolFromSmiles(chemical_smiles)
print(mol)

ERROR-type MOFIDs: 1691
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.ERROR
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn]
None


[11:53:00] Explicit valence for atom # 89 O, 3, is greater than permitted


In [8]:
# Now test all 1,691 ERROR records with RDkit : 
error_mofids = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"]

parsed = 0
failed = 0

for mofid in error_mofids:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
    else:
        parsed += 1

print("Total ERROR records:", len(error_mofids))
print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

[11:53:00] Explicit valence for atom # 89 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 25 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 33 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 23 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 13 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 31 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 101 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 33 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 63 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 51 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 55 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 54 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 51 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom 

Total ERROR records: 1691
Successfully parsed: 1628
Failed to parse: 63


[11:53:00] Explicit valence for atom # 19 N, 4, is greater than permitted
[11:53:00] Explicit valence for atom # 4 N, 4, is greater than permitted
[11:53:00] Explicit valence for atom # 3 N, 4, is greater than permitted
[11:53:00] Explicit valence for atom # 137 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 89 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 12 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 37 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 37 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 78 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 31 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 41 O, 3, is greater than permitted


---

*27,706 non-null MOFID rows → remove 1,754 NA placeholders → 25,952 rows → 1,691 ERROR-labelled records → 1,628 parse successfully and only 63 fail.*

---




### Can RDKit parse the chemical representation for all 25,952 remaining records?

In [9]:
print("Dataset shape:", df.shape)

Dataset shape: (25952, 11)


In [10]:
# first look at one MOFID from  dataset:
mofid = df["mofid"].iloc[0]

print(mofid)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0


---
I has two part chemical representation and MOFid metatdata:  `MOFid-v1.pcu.cat0`

---

In [11]:
# .split(" ") separates the MOFID string at the space.
# [0] selects the chemical representation before the MOFID metadata.
chemical_smiles = mofid.split(" ")[0]

print(chemical_smiles)

from rdkit import Chem

mol = Chem.MolFromSmiles(chemical_smiles)

print(mol)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]
None


[11:53:00] Explicit valence for atom # 13 O, 3, is greater than permitted


----
whether we can isolate the linker from your MOFID and process that linker with RDKit.

----

In [12]:
fragments = chemical_smiles.split(".")

print(fragments)
fragment = Chem.MolFromSmiles(fragments[0])

print(fragment)

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']


---
<rdkit.Chem.rdchem.Mol object at 0x75c18c10e8f0>

RDKit successfully parsed the first fragment.

---

In [13]:
# Check whole dataset with RDKit 
failed_mofids = []
parsed = 0 # number RDKit successfully reads
failed = 0 # number RDKit cannot read
for mofid in df["mofid"]:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
        failed_mofids.append(mofid)   
    else:
        parsed += 1

print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

[11:53:00] Explicit valence for atom # 13 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 13 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 16 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 31 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 27 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 30 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 24 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 57 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 57 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 27 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 65 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 69 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom # 94 O, 3, is greater than permitted
[11:53:00] Explicit valence for atom #

Successfully parsed: 12940
Failed to parse: 13012


[11:53:07] Explicit valence for atom # 69 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 69 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 68 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 68 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 42 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 70 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 70 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 83 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 83 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 87 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 87 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 77 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom # 77 O, 3, is greater than permitted
[11:53:07] Explicit valence for atom #

---
Can standard RDKit parse all 25,952 complete chemical representations?

No. It parses 12,940, while 13,012 fail.

Next Step: should be only to collect the failed MOFIDs so we can inspect what causes these failures.

----

In [14]:
# collect failed mofid to understand about their features:
# We check a small batch of dataset
for mofid in failed_mofids[:10]:
    print(mofid)
    print()

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C(=O)C(=O)[O-].[O-]C(=O)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.UNKNOWN.cat0

[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

-----

This supports hypothesis that RDKit may be rejecting the Zn–O coordination fragment, while the organic linker itself may still be readable.

-----

In [15]:
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?

zn_o_cluster = "[Zn][O]([Zn])([Zn])[Zn]"

with_zn_o_cluster = 0
without_zn_o_cluster = 0

for mofid in failed_mofids:
    if zn_o_cluster in mofid:
        with_zn_o_cluster += 1
    else:
        without_zn_o_cluster += 1

print(
    f"With Zn-O cluster: {with_zn_o_cluster}, "
    f"Without Zn-O cluster: {without_zn_o_cluster}"
)

With Zn-O cluster: 12284, Without Zn-O cluster: 728


In [16]:
without_zn_o_cluster = []
for mofid in failed_mofids:
    if zn_o_cluster not in mofid:
        without_zn_o_cluster.append(mofid)

for mofid in without_zn_o_cluster[:10]:
    print(mofid)
    print() 

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(cc1O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat1

CCOC1=[N]=C(C=N[CH]1)OCC.CCOc1cc(cc(c1C(=O)[O-])OCC)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].CCCOC1=[N]=C(C(=N[CH]1)OCCC)OCCC.[O-]C(=O)c1ccc(cc1OCCC)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

COC1=[N]=C(OC)C=C([CH]1)c1ccncc1OC.COc1cc(C(=O)[O-]

In [17]:
# get overall overview of data :
error_count = 0
pcu_cat0_count = 0
pcu_cat1_count = 0
n2_count = 0

for mofid in failed_mofids:
    if "ERROR" in mofid:
        error_count += 1
    if "pcu.cat0" in mofid:
        pcu_cat0_count += 1
    if "pcu.cat1" in mofid:
        pcu_cat1_count += 1
    if "N#N" in mofid:
        n2_count += 1

print("ERROR:", error_count)
print("pcu.cat0:", pcu_cat0_count)
print("pcu.cat1:", pcu_cat1_count)
print("Contains N#N:", n2_count)

ERROR: 63
pcu.cat0: 7210
pcu.cat1: 3659
Contains N#N: 6


| Pattern        | Failed MOFIDs |
| -------------- | ------------: |
| `ERROR`        |            63 |
| `pcu.cat0`     |         7,210 |
| `pcu.cat1`     |         3,659 |
| Contains `N#N` |             6 |


In [18]:
# fragment analysis and Linker extraction:
# Step 1 :Extract the chemical portion of each MOFID by removing the MOFid-v1... metadata.
chemical_representations = []

for mofid in df["mofid"]:
    chemical_smiles = mofid.split("MOFid-v1")[0].strip() # space and mofide_v1“For every MOFID, remove the MOFID metadata and store only its chemical representation.”
    chemical_representations.append(chemical_smiles)

for chemical in chemical_representations[:5]:
    print(chemical)
    print()


[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]



In [19]:
df["chemical_representation"] = chemical_representations
print(df.columns)
df_fragments = df.copy()

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation'],
      dtype='object')


In [20]:
# Metal Discconector
from rdkit.Chem.MolStandardize import rdMolStandardize
chemical = df["chemical_representation"].iloc[0]
# Create RDKit molecule without standard sanitization
mof_mol = Chem.MolFromSmiles(chemical, sanitize=False)
mof_mol.UpdatePropertyCache(strict=False)
print(mof_mol)
# Initialize MetalDisconnector
disconnector = rdMolStandardize.MetalDisconnector()
# Disconnect metal-nonmetal bonds
disconnected_mol = disconnector.Disconnect(mof_mol)


[11:53:07] Initializing MetalDisconnector
[11:53:07] Running MetalDisconnector
[11:53:07] Removed covalent bond between Zn and O
[11:53:07] Removed covalent bond between Zn and O
[11:53:07] Removed covalent bond between Zn and O
[11:53:07] Removed covalent bond between Zn and O


In [21]:
# Split the disconnected MOF into individual molecular fragments
fragments = Chem.GetMolFrags(disconnected_mol, asMols=True)

# Print the total number of fragments obtained
print(len(fragments))

# Convert each RDKit fragment back to a SMILES string and display it
for fragment in fragments:
    print(Chem.MolToSmiles(fragment))


6
O=C([O-])c1ccc(C(=O)[O-])cc1
[Zn+]
[O-2]
[Zn+]
[Zn]
[Zn]


In [22]:
# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)
# Define SMARTS for unwanted inorganic nodes/solvents
inorganic_blacklist = [
    Chem.MolFromSmarts('[O-2]'),       # Oxide ion bridges
    Chem.MolFromSmarts('N#N'),         # Nitrogen gas
    Chem.MolFromSmarts('[O;H2]'),       # Water
    Chem.MolFromSmarts('[O-]S(=O)(=O)[O-]') # Sulfate (if applicable)
]
# Create an empty list for fragments that are not metal atoms
non_metal_fragments = []

# Examine each fragment from the first MOF
for frag in fragments:

    # If the fragment contains exactly one atom
    if frag.GetNumAtoms() == 1:

        # Get that atom
        atom = frag.GetAtomWithIdx(0)

        # If that atom is a metal, skip it
        if atom.GetAtomicNum() in metal_atomic_numbers:
            continue
      # Check if the fragment matches any blacklisted substructure completely
    is_inorganic = False
    for pattern in inorganic_blacklist:
        if frag.HasSubstructMatch(pattern):
            # Ensure the match isn't just a part of a larger organic molecule
            if frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms():
                is_inorganic = True
                break
                
    if not is_inorganic:
    # Keep everything that was not removed as a metal
        non_metal_fragments.append(frag)

# Display the fragments remaining after metal removal
for frag in non_metal_fragments:
    print(Chem.MolToSmiles(frag))

O=C([O-])c1ccc(C(=O)[O-])cc1


In [33]:
# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)

# Define SMARTS for unwanted inorganic nodes/solvents
inorganic_blacklist = [
    Chem.MolFromSmarts('[O-2]'),
    Chem.MolFromSmarts('N#N'),
    Chem.MolFromSmarts('[O;H2]'),
    Chem.MolFromSmarts('[O-]S(=O)(=O)[O-]')
]


def extract_linkers(chemical):
    """Return valid organic linker SMILES and skip invalid fragments."""
    lig_mol = Chem.MolFromSmiles(chemical, sanitize=False)
    if lig_mol is None:
        raise ValueError("RDKit could not create a molecule from the chemical representation")

    lig_mol.UpdatePropertyCache(strict=False)
    disconnected_mol = rdMolStandardize.MetalDisconnector().Disconnect(lig_mol)
    fragments = Chem.GetMolFrags(disconnected_mol, asMols=True)

    linker_smiles = []
    invalid_fragments = []

    for frag in fragments:
        if frag.GetNumAtoms() == 1:
            atom = frag.GetAtomWithIdx(0)
            if atom.GetAtomicNum() in metal_atomic_numbers:
                continue

        is_inorganic = any(
            frag.HasSubstructMatch(pattern)
            and frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms()
            for pattern in inorganic_blacklist
        )
        if is_inorganic:
            continue

        try:
            Chem.SanitizeMol(frag)
            linker_smiles.append(Chem.MolToSmiles(frag))
        except Exception as error:
            invalid_fragments.append({
                "fragment": Chem.MolToSmiles(frag, sanitize=False),
                "error": str(error),
            })

    return linker_smiles, invalid_fragments

In [30]:
extract_linkers(df["chemical_representation"].iloc[0])

[12:07:48] Initializing MetalDisconnector
[12:07:48] Running MetalDisconnector
[12:07:48] Removed covalent bond between Zn and O
[12:07:48] Removed covalent bond between Zn and O
[12:07:48] Removed covalent bond between Zn and O
[12:07:48] Removed covalent bond between Zn and O


['O=C([O-])c1ccc(C(=O)[O-])cc1']

In [ ]:
# Store extracted linkers and invalid-fragment diagnostics for every MOF
all_linkers = []
failed_linker_extraction = []

for index, chemical in df["chemical_representation"].items():
    try:
        linkers, invalid_fragments = extract_linkers(chemical)
        all_linkers.append(linkers)

        if invalid_fragments:
            failed_linker_extraction.append({
                "index": index,
                "chemical_representation": chemical,
                "invalid_fragments": invalid_fragments,
            })

    except Exception as error:
        print(f"Failed at row {index}: {error}")
        all_linkers.append([])
        failed_linker_extraction.append({
            "index": index,
            "chemical_representation": chemical,
            "error": str(error),
        })

df["linker_smiles"] = all_linkers

print("MOFs with invalid or failed fragments:", len(failed_linker_extraction))

[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond b

Failed at row 25: Explicit valence for atom # 17 N, 4, is greater than permitted
Chemical representation: [O-]C(=O)C(=O)[O-].[O-]C(=O)C1=C(N)C2C3[CH]C(=C(C45C([C]1N)(C2C34)[NH2][NH2]5)N)C(=O)[O-].[O-]C(=O)C1=CC23C4C5([CH]1)[NH2]C15C4C3([NH2]2)C=C([CH]1)C(=O)[O-].[O-]C(=O)c1cc2cc(N)c(cc2cc1N)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2N)C(=O)[O-])N.[Zn][O]([Zn])([Zn])[Zn]
Failed at row 26: Explicit valence for atom # 10 N, 4, is greater than permitted
Chemical representation: [O-]C(=O)C(=O)[O-].[O-]C(=O)C1=C(N)C2(C34C5([CH]1)[NH2]C15[C](N)C([NH2]4)([C](C2C31)N)C(=O)[O-])N.[O-]C(=O)C1=C(N)C2C34C([CH]1)C1C3C2(N)[C](C([C]1N)([NH2]4)C(=O)[O-])N.[O-]C(=O)c1cc2c(N)c(N)c(c(c2cc1N)N)C(=O)[O-].[O-]C(=O)c1cc2ccc(c(c2c(c1N)N)N)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]


[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Initializing MetalDisconnector
[12:15:48] Running MetalDisconnector
[12:15:48] Removed covalent bond between Zn and O
[12:15:48] Removed covalent bond b

Failed at row 338: Explicit valence for atom # 14 N, 4, is greater than permitted
Chemical representation: Nc1cc(cc2c1-c1cc(N)c(cc1[NH2][NH2]2)C(=O)[O-])C(=O)[O-].Nc1ccc2c(c1)c(cc(c2C(=O)[O-])N)C(=O)[O-].[O-]C(=O)c1c(N)c(N)c(c2c1c(N)ccc2)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]


[12:15:49] Initializing MetalDisconnector
[12:15:49] Running MetalDisconnector
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Initializing MetalDisconnector
[12:15:49] Running MetalDisconnector
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Initializing MetalDisconnector
[12:15:49] Running MetalDisconnector
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Removed covalent bond between Zn and O
[12:15:49] Re

...

Failed at row 27218: Explicit valence for atom # 17 O, 3, is greater than permitted
Chemical representation: [O-]C(=O)[C]=C([Br]1[O]2C3=C[CH]C(=[C][C]4O[Zn]56[O]78[Zn]92[O]21[C](C=C[C](Br)C(=C1O[Zn]7(O3)O[C](O6)[C]=C([CH]C=C([O]9[Br]2C(=[C]C(=O)[O-])C=CC(=O)[O-])O[Zn]8(O4)O1)Br)Br)O5)Br)C=CC(=O)[O-].[O-]C(=O)C=CC(=C(C(=O)[O-])Br)Br.[O-]C(=O)C=CC(=C(C(=O)[O-])Br)Br.[O-]C(=O)C=CC(=[C]C(=O)[O-])Br.[O-]C(=O)C=CC(=[C]C(=O)[O-])Br.[Zn][O]([Zn])([Zn])[Zn]


[12:16:59] Initializing MetalDisconnector
[12:16:59] Running MetalDisconnector
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Initializing MetalDisconnector
[12:16:59] Running MetalDisconnector
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Initializing MetalDisconnector
[12:16:59] Running MetalDisconnector
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Initializing MetalDisconnector
[12:16:59] Running MetalDisconnector
[12:16:59] Removed covalent bond between Zn and O
[12:16:59] Removed covalent bond b

Failed at row 27393: Explicit valence for atom # 7 N, 4, is greater than permitted
Chemical representation: [O-]C(=O)[CH]C12C3=C4[N]52C1([C][N]C1C2=C(C(=O)[O-])C67C(=C1C(=O)[O-])[C][N]C1C(=CC([N][C]C8=C(C4([N]3)C3=C(C8[N][C]C48C([N]98C4([CH]C(=O)[O-])C4=C9C8([N]4)C(=C([C][N]C4C(=CC([N][C]3)C(=C4)C(=O)[O-])C(=O)[O-])C([N][C]C34[N]9(C7=C([N]6)C39[CH]C(=O)[O-])C4C(=O)[O-])C(=C8[C][N]C3C(=CC([N][C]2)C(=C3)C(=O)[O-])C(=O)[O-])C(=O)[O-])C(=O)[O-])C(=O)[O-])C(=O)[O-])C(=O)[O-])C(=C1)C(=O)[O-])C(=O)[O-])C5C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]


[12:17:00] Initializing MetalDisconnector
[12:17:00] Running MetalDisconnector
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Initializing MetalDisconnector
[12:17:00] Running MetalDisconnector
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Initializing MetalDisconnector
[12:17:00] Running MetalDisconnector
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Initializing MetalDisconnector
[12:17:00] Running MetalDisconnector
[12:17:00] Removed covalent bond between Zn and O
[12:17:00] Removed covalent bond b

Failed extractions: 847
